# 🚀 TensorFlow Object Detection — SSD MobileNet V2 FPNLite 320×320
### Optimized for Raspberry Pi & Low-Power Edge Devices

---

## 📋 Overview

This notebook trains a **lightweight object detection model** using Google's TensorFlow Object Detection API.
The end result is a model small enough to run on a **Raspberry Pi**, a microcontroller, or any device with limited RAM and no GPU.

### Why SSD MobileNet V2 FPNLite 320×320?

| Property | Value |
|---|---|
| Architecture | Single Shot Detector (SSD) |
| Backbone | MobileNet V2 (depthwise-separable convolutions) |
| Neck | FPN Lite (Feature Pyramid Network — lightweight version) |
| Input resolution | 320 × 320 px |
| Size on disk | ~22 MB (float32) |
| TFLite int8 size | ~3–5 MB |
| COCO mAP | ~22 (good enough for single-class or few-class tasks) |

**Why NOT a bigger model?**
- EfficientDet D1+ requires >500 MB RAM during inference — a Raspberry Pi 4 only has 1–4 GB shared with the OS.
- Faster RCNN and DETR use two-stage detection, which is too slow for real-time edge use.
- SSD + MobileNet V2 was designed from the start to be hardware-efficient using **depthwise separable convolutions** that drastically reduce multiply-accumulate operations (MACs).

---

## 🗺️ Notebook Sections

1. Environment Setup & GPU Check
2. Dependency Installation (protobuf fix, TFOD API)
3. TensorFlow Models Repository Setup
4. Protobuf Compilation
5. Object Detection API Installation & Verification
6. Download Pretrained SSD MobileNet V2 FPNLite Checkpoint
7. Dataset Paths & Verification
8. Edit `pipeline.config` for Fine-tuning
9. Train the Model (with Resume Support)
10. Export the Trained Model
11. Test Inference on Images
12. Convert to TensorFlow Lite (Float16 & Int8)
13. Raspberry Pi Deployment Tips


---
## 1️⃣ Environment Setup & GPU Check

Kaggle provides a free GPU (NVIDIA Tesla T4 or P100). We verify it is available and
check the TensorFlow version so we know what to expect during training.

> **Note:** Go to *Settings → Accelerator → GPU T4 x2* on the right sidebar before running this notebook.

In [ ]:
import os
import sys

import tensorflow as tf

# Print TensorFlow version — we need ≥ 2.8 for FPNLite support
print("TensorFlow version:", tf.__version__)

# List available GPUs. If the list is empty the notebook will still work but training will be slow.
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)

# Allow GPU memory growth instead of allocating all memory at once.
# This prevents OOM (Out-Of-Memory) crashes when multiple processes share the GPU.
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("GPU memory growth enabled — Kaggle GPU is ready.")

---
## 2️⃣ Dependency Installation

### Why this section is important

The TensorFlow Object Detection API (TFOD) is **not** part of the standard `tensorflow` pip package.
It lives in Google's `models` repository and must be installed separately.

### The Protobuf Version Problem

TFOD uses `.proto` files to define its configuration format (`pipeline.config`).
These are compiled into Python classes with `protoc`.

**Problem:** `protobuf >= 4.x` introduced a breaking API change that makes older TFOD code crash with:
```
TypeError: Descriptors cannot not be created directly.
```

**Fix:** Pin `protobuf` to `3.20.x` which is the last stable version compatible with TFOD's generated `.pb2.py` files.
This is the *official* recommended fix from the TensorFlow team.

In [ ]:
# ── Protobuf fix: pin to 3.20.x to avoid breaking API change in protobuf ≥ 4 ──
# This MUST be installed before anything else imports google.protobuf.
!pip install -q "protobuf==3.20.3"

# ── Other TFOD API runtime dependencies ──
# tf-slim: high-level TF library used by many detection model definitions
# lvis: evaluation metrics library (required for COCO-style evaluation)
# scipy: needed by some augmentation utilities
# tf-models-official: brings in some shared model utilities
!pip install -q tf-slim lvis scipy

# ── pycocotools: COCO metric computation ──
# Used during evaluation to compute mAP, mAP@50, mAP@75 etc.
!pip install -q pycocotools

print("✅ Dependencies installed.")

---
## 3️⃣ TensorFlow Models Repository Setup

The TFOD API is part of Google's `models` repository on GitHub.
We clone it to `/tmp/models` so it does not pollute the working directory.

### What is inside `models/research/object_detection`?

- `builders/` — Factory functions that build model graphs from `pipeline.config`
- `core/` — Loss functions, anchor generators, post-processing
- `models/` — Architecture definitions (SSD, Faster RCNN, CenterNet…)
- `protos/` — Protobuf `.proto` schema files
- `utils/` — Visualization helpers, TFRecord utilities
- `model_main_tf2.py` — The main training script we will call
- `exporter_main_v2.py` — The export script that produces a SavedModel

In [ ]:
import os

MODELS_DIR = "/tmp/models"  # Clone destination — /tmp is writable on Kaggle

if not os.path.exists(MODELS_DIR):
    # Clone only the latest commit (--depth 1) to save time and disk space.
    # The full git history of google/models is huge (~2 GB) and we don't need it.
    !git clone --depth 1 https://github.com/tensorflow/models {MODELS_DIR}
    print("✅ Repository cloned to", MODELS_DIR)
else:
    print("ℹ️  Repository already exists at", MODELS_DIR, "— skipping clone.")

---
## 4️⃣ Protobuf Compilation

### What is `protoc`?

`protoc` is the **Protocol Buffer Compiler**. It reads `.proto` schema files and generates
Python classes (`*_pb2.py`) that allow Python code to read/write those binary messages.

TFOD's `pipeline.config` format is defined in those `.proto` files.
Until we compile them, Python does not know how to parse a `pipeline.config` file.

### What happens behind the scenes?

```
models/research/object_detection/protos/pipeline.proto
         ↓  protoc
models/research/object_detection/protos/pipeline_pb2.py  ← generated Python class
```

Every `.proto` file in the `protos/` directory gets compiled.

In [ ]:
import os

RESEARCH_DIR = "/tmp/models/research"

# Change into the research directory — protoc paths are relative to here
os.chdir(RESEARCH_DIR)

# Compile ALL .proto files inside object_detection/protos/
# The output *_pb2.py files are written next to the source .proto files.
# The --python_out=. flag means: write Python output relative to current directory.
!protoc object_detection/protos/*.proto --python_out=.

# Quick sanity check — count how many pb2.py files were generated
pb2_files = [f for f in os.listdir("object_detection/protos") if f.endswith("_pb2.py")]
print(f"✅ Compiled {len(pb2_files)} protobuf schema files.")

---
## 5️⃣ Object Detection API Installation & Verification

We now install TFOD as a Python package using its `setup.py` script.
This makes `import object_detection` work from any directory.

After installation we run the official unit tests (`model_builder_tf2_test.py`)
to confirm everything is wired correctly.

> **Why test?** Silent import errors are common with TFOD.
> Running this test catches protobuf mismatches and missing dependencies early.

In [ ]:
import os

RESEARCH_DIR = "/tmp/models/research"
os.chdir(RESEARCH_DIR)

# Copy the TFOD setup.py from its subdirectory to research/ and install it.
# This registers the `object_detection` package in the current Python environment.
!cp object_detection/packages/tf2/setup.py .
!pip install -q -e .

# Add slim to the Python path — some model definitions still import from slim directly
sys.path.append(f"{RESEARCH_DIR}/slim")

print("✅ Object Detection API installed.")

In [ ]:
import sys
RESEARCH_DIR = "/tmp/models/research"

# Run the official model builder test.
# This verifies that all protobuf-generated classes are importable and that
# the SSD / Faster-RCNN model builder works correctly.
# Expected output: ... OK (no FAILED or ERROR lines)
!python {RESEARCH_DIR}/object_detection/builders/model_builder_tf2_test.py

---
## 6️⃣ Download Pretrained SSD MobileNet V2 FPNLite Checkpoint

### Transfer Learning — Why We Start From a Pretrained Checkpoint

Training an object detector from random weights on a small custom dataset almost always
produces poor results. The model needs to first learn low-level visual features
(edges, textures, shapes) before it can learn high-level concepts ("human", "car").

**Transfer Learning** solves this:
1. Start from a checkpoint pre-trained on **COCO** (330k images, 80 classes).
2. The backbone already knows how to extract useful features.
3. We only need to **fine-tune** the detection head for our specific classes.

This requires far fewer epochs and far less data.

### What is in the downloaded `.tar.gz`?

```
ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/
  ├── checkpoint/
  │   ├── ckpt-0.index       ← index of variables in the checkpoint
  │   ├── ckpt-0.data-00000-of-00001  ← actual weight values
  ├── pipeline.config        ← reference configuration for this model
  └── saved_model/           ← full SavedModel (optional, for inference)
```

We use the `checkpoint/` folder to warm-start training.

In [ ]:
import os
import tarfile
import urllib.request

# ── Paths ──────────────────────────────────────────────────────────────────────
PRETRAINED_DIR = "/tmp/pretrained_models"
os.makedirs(PRETRAINED_DIR, exist_ok=True)

MODEL_NAME    = "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"
DOWNLOAD_URL  = (
    "http://download.tensorflow.org/models/object_detection/tf2/20200711/"
    f"{MODEL_NAME}.tar.gz"
)
TAR_PATH      = os.path.join(PRETRAINED_DIR, f"{MODEL_NAME}.tar.gz")
EXTRACT_PATH  = os.path.join(PRETRAINED_DIR, MODEL_NAME)

# ── Download (skip if already present) ────────────────────────────────────────
if not os.path.exists(EXTRACT_PATH):
    print(f"Downloading {MODEL_NAME} ...")
    urllib.request.urlretrieve(DOWNLOAD_URL, TAR_PATH)

    # Extract the tarball
    print("Extracting archive...")
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        tar.extractall(PRETRAINED_DIR)

    print("✅ Pretrained checkpoint ready at:", EXTRACT_PATH)
else:
    print("ℹ️  Pretrained model already exists — skipping download.")

# Path to the checkpoint prefix used when warm-starting training
PRETRAINED_CKPT = os.path.join(EXTRACT_PATH, "checkpoint", "ckpt-0")
# Path to the reference pipeline.config bundled with this checkpoint
REFERENCE_CONFIG = os.path.join(EXTRACT_PATH, "pipeline.config")

print("Checkpoint prefix :", PRETRAINED_CKPT)
print("Reference config  :", REFERENCE_CONFIG)

---
## 7️⃣ Dataset Paths & Verification

### Dataset Layout

```
/kaggle/input/datasets/alluketansatyasai/humans-original-tfrecord/
  ├── train/
  │   ├── humans.tfrecord
  │   └── humans_label_map.pbtxt
  ├── valid/
  │   ├── humans.tfrecord
  │   └── humans_label_map.pbtxt
  └── test/
      ├── humans.tfrecord
      └── humans_label_map.pbtxt
```

### What is a TFRecord?

A TFRecord is a **binary file format** used by TensorFlow to store
large datasets efficiently. Each record encodes:
- The raw JPEG bytes of an image
- Bounding box coordinates (xmin, ymin, xmax, ymax) in normalized [0,1] space
- Class labels as integer IDs and/or string names

TFRecords are read with `tf.data` pipelines, which support:
- Parallel reading from disk
- On-the-fly augmentation
- Prefetching to overlap I/O and GPU computation

### What is a label map (`.pbtxt`)?

A label map maps integer IDs to human-readable class names:
```
item {
  id: 1
  name: 'human'
}
```
TFOD internally uses these IDs during training and evaluation.

In [ ]:
import os

# ── Dataset root on Kaggle ─────────────────────────────────────────────────────
DATASET_ROOT = "/kaggle/input/datasets/alluketansatyasai/humans-original-tfrecord"

TRAIN_RECORD   = os.path.join(DATASET_ROOT, "train",  "humans.tfrecord")
VALID_RECORD   = os.path.join(DATASET_ROOT, "valid",  "humans.tfrecord")
TEST_RECORD    = os.path.join(DATASET_ROOT, "test",   "humans.tfrecord")
LABEL_MAP_PATH = os.path.join(DATASET_ROOT, "train",  "humans_label_map.pbtxt")

# ── Verify all required files exist ───────────────────────────────────────────
required_files = {
    "Train TFRecord"  : TRAIN_RECORD,
    "Valid TFRecord"  : VALID_RECORD,
    "Test TFRecord"   : TEST_RECORD,
    "Label Map"       : LABEL_MAP_PATH,
}

all_ok = True
for name, path in required_files.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌ MISSING"
    print(f"{status}  {name:20s} → {path}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        "One or more dataset files are missing. "
        "Make sure the dataset is added via: "
        "Add Data → Search 'humans-original-tfrecord'"
    )

print("\n✅ All dataset files verified.")

In [ ]:
import tensorflow as tf

# ── Count records in each split ────────────────────────────────────────────────
# Iterating through a TFDataset without decoding is fast — we just count
# the raw serialized examples to confirm how many images are in each split.

def count_tfrecord(path: str) -> int:
    """Count the number of examples in a TFRecord file."""
    dataset = tf.data.TFRecordDataset(path)
    # sum(1 for _ in dataset) is memory-efficient — we never decode the images
    return sum(1 for _ in dataset)

print("Counting records (this may take ~30 s for large datasets)...")
n_train = count_tfrecord(TRAIN_RECORD)
n_valid = count_tfrecord(VALID_RECORD)
n_test  = count_tfrecord(TEST_RECORD)

print(f"  Train  : {n_train:,} images")
print(f"  Valid  : {n_valid:,} images")
print(f"  Test   : {n_test:,} images")
print(f"  Total  : {n_train + n_valid + n_test:,} images")

In [ ]:
# ── Read and display the label map ─────────────────────────────────────────────
with open(LABEL_MAP_PATH, "r") as f:
    label_map_str = f.read()

print("Label map contents:")
print(label_map_str)

# Parse the label map with the TFOD utility so we can count classes
from object_detection.utils import label_map_util

label_map = label_map_util.load_labelmap(LABEL_MAP_PATH)
categories = label_map_util.convert_label_map_to_categories(
    label_map, max_num_classes=100
)

NUM_CLASSES = len(categories)
print(f"\nDetected {NUM_CLASSES} class(es): {[c['name'] for c in categories]}")

---
## 8️⃣ Edit `pipeline.config` for Fine-Tuning

### What is `pipeline.config`?

`pipeline.config` is the **single source of truth** for the entire training run.
It is a Protocol Buffer text file that specifies:

| Section | What it controls |
|---|---|
| `model {}` | Architecture: SSD, anchor sizes, num_classes |
| `train_config {}` | Batch size, learning rate schedule, warm-start checkpoint |
| `train_input_reader {}` | Path to training TFRecord and label map |
| `eval_config {}` | Evaluation metrics and frequency |
| `eval_input_reader {}` | Path to validation TFRecord |

### Key values we customize

- **`num_classes`** — must match our label map (e.g., 1 for humans only)
- **`batch_size`** — set to 8 to avoid OOM on Kaggle's 16 GB GPU
- **`fine_tune_checkpoint`** — path to our downloaded COCO checkpoint
- **`fine_tune_checkpoint_type`** — `"detection"` means we restore *all* weights
  (backbone + neck + head) and only retrain on our data
- **`num_steps`** — total training steps (not epochs); 5000–10000 is usually sufficient for
  a single-class fine-tune
- **`label_map_path`** and **`input_path`** — absolute paths to our Kaggle dataset

> **Tip:** Increasing `batch_size` speeds up training but raises memory usage.
> For Kaggle's T4 GPU (16 GB), `batch_size=8` is safe for 320×320 images.

In [ ]:
import os
import shutil

# ── Configuration ──────────────────────────────────────────────────────────────
TRAINING_DIR = "/tmp/training"
os.makedirs(TRAINING_DIR, exist_ok=True)

# We copy the reference pipeline.config to our writable training directory
# so we can modify it without touching the read-only /tmp/pretrained_models/ copy.
PIPELINE_CONFIG = os.path.join(TRAINING_DIR, "pipeline.config")
shutil.copy(REFERENCE_CONFIG, PIPELINE_CONFIG)

print("✅ Copied reference config to:", PIPELINE_CONFIG)

In [ ]:
# ── Programmatically edit the pipeline.config ──────────────────────────────────
# We use TFOD's pipeline_pb2 and text_format tools to parse and update the
# config as a proper protobuf object rather than fragile string replacement.

from google.protobuf import text_format
from object_detection.protos import pipeline_pb2

# Load the config file into a Python protobuf object
pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()
with open(PIPELINE_CONFIG, "r") as f:
    text_format.Merge(f.read(), pipeline_config)

# ── 1. Number of classes ───────────────────────────────────────────────────────
# Must match the number of items in the label map.
pipeline_config.model.ssd.num_classes = NUM_CLASSES

# ── 2. Batch size ─────────────────────────────────────────────────────────────
# 8 is memory-safe on Kaggle T4 GPU (16 GB VRAM) for 320x320 images.
# Increase to 16 if you have headroom; decrease to 4 if you hit OOM.
pipeline_config.train_config.batch_size = 8

# ── 3. Warm-start checkpoint ───────────────────────────────────────────────────
# Tell TF where to load the pre-trained COCO weights from.
pipeline_config.train_config.fine_tune_checkpoint = PRETRAINED_CKPT

# "detection" = restore the full detection model (backbone + head).
# Use "classification" only when restoring a backbone-only checkpoint.
pipeline_config.train_config.fine_tune_checkpoint_type = "detection"

# ── 4. Use v2 checkpoint format ────────────────────────────────────────────────
# TF2 checkpoints use a different variable naming scheme than TF1.
pipeline_config.train_config.use_bfloat16 = False  # True only for TPUs

# ── 5. Total training steps ────────────────────────────────────────────────────
# 10 000 steps is a good starting point for a single-class fine-tune.
# Each step processes one mini-batch. With batch_size=8 and 1000 training images,
# 10 000 steps ≈ 80 epochs.
pipeline_config.train_config.num_steps = 10000

# ── 6. Training data paths ─────────────────────────────────────────────────────
pipeline_config.train_input_reader.label_map_path = LABEL_MAP_PATH
pipeline_config.train_input_reader.tf_record_input_reader.input_path[:] = [TRAIN_RECORD]

# ── 7. Validation data paths ───────────────────────────────────────────────────
pipeline_config.eval_input_reader[0].label_map_path = LABEL_MAP_PATH
pipeline_config.eval_input_reader[0].tf_record_input_reader.input_path[:] = [VALID_RECORD]

# ── Write the modified config back to disk ─────────────────────────────────────
with open(PIPELINE_CONFIG, "w") as f:
    f.write(text_format.MessageToString(pipeline_config))

print("✅ pipeline.config updated and saved to:", PIPELINE_CONFIG)
print(f"   num_classes      = {pipeline_config.model.ssd.num_classes}")
print(f"   batch_size       = {pipeline_config.train_config.batch_size}")
print(f"   num_steps        = {pipeline_config.train_config.num_steps}")
print(f"   checkpoint       = {pipeline_config.train_config.fine_tune_checkpoint}")

---
## 9️⃣ Train the Model

### What happens during training?

1. **Data pipeline** (`tf.data`): TFRecord files are decoded, images are resized to 320×320,
   bounding boxes are normalized, and random augmentations are applied (horizontal flip,
   random crop, color jitter).

2. **Forward pass**: The image batch goes through:
   - MobileNet V2 backbone → feature maps at multiple scales
   - FPN Lite neck → fuses multi-scale features
   - SSD prediction head → outputs class logits + box offsets for each anchor

3. **Loss computation**:
   - **Classification loss**: Focal loss (hard example mining)
   - **Localization loss**: Smooth L1 (Huber) loss on matched anchors

4. **Backpropagation**: Gradients flow back through the whole network;
   Adam optimizer updates the weights.

5. **Checkpointing**: Every 1000 steps (configurable) the model saves a checkpoint to
   `TRAINING_DIR/`. Checkpoints contain all trainable variable values.
   If training is interrupted, we can resume from the latest checkpoint.

### Resume Training Support

`model_main_tf2.py` automatically resumes from the latest checkpoint in `model_dir`
if one exists. There is no special flag needed — just run the same command again.

### Monitoring with TensorBoard

TensorBoard summaries are written to `TRAINING_DIR/train/` and `TRAINING_DIR/eval/`.
You can launch TensorBoard in a separate Kaggle cell while training runs:
```
%load_ext tensorboard
%tensorboard --logdir /tmp/training
```

In [ ]:
import os

RESEARCH_DIR  = "/tmp/models/research"
TRAINING_DIR  = "/tmp/training"

# model_main_tf2.py is the official TFOD training entry point for TF2.
# Key flags:
#   --model_dir       : where to save checkpoints and TensorBoard summaries
#   --pipeline_config_path : the pipeline.config we edited above
#   --num_train_steps : overrides the value in the config (optional, for quick tests)
#   --alsologtostderr : print progress to the Kaggle cell output

!python {RESEARCH_DIR}/object_detection/model_main_tf2.py \
    --model_dir={TRAINING_DIR} \
    --pipeline_config_path={PIPELINE_CONFIG} \
    --num_train_steps=10000 \
    --alsologtostderr 2>&1 | tail -100

# ▲ We pipe through `tail -100` to keep cell output manageable.
#   Remove that if you want to see every log line.

print("\n✅ Training complete (or interrupted). Checkpoint saved to:", TRAINING_DIR)

### 🔄 Resume Training (Run this cell to add more steps)

If you need more training steps (e.g., the loss is still high), increase `num_steps`
in the config and re-run the cell above. TFOD will automatically detect and restore
the latest checkpoint.

In [ ]:
# ── List saved checkpoints ─────────────────────────────────────────────────────
# The checkpoint file contains a JSON-like map of step number → file prefix.
# The latest checkpoint is the one with the highest step number.

import os

ckpt_files = [f for f in os.listdir(TRAINING_DIR) if f.startswith("ckpt-")]
print("Checkpoints found:")
for f in sorted(ckpt_files):
    full = os.path.join(TRAINING_DIR, f)
    size_mb = os.path.getsize(full) / 1e6
    print(f"  {f:<40s}  ({size_mb:.1f} MB)")

---
## 🔟 Export the Trained Model

### Checkpoint vs. SavedModel vs. TFLite

| Format | Use case |
|---|---|
| **Checkpoint** (`.ckpt`) | Resuming training; contains optimizer state |
| **SavedModel** (`saved_model/`) | Serving with TF Serving, Python inference; no optimizer state |
| **TFLite** (`.tflite`) | On-device deployment: Android, iOS, Raspberry Pi |

`exporter_main_v2.py` reads the latest checkpoint from `TRAINING_DIR` and writes
a clean **SavedModel** to `EXPORT_DIR`. This SavedModel:
- Strips the optimizer state (smaller file)
- Freezes the graph (replaces `tf.Variable` with constants)
- Wraps the pre/post-processing steps so you can call it with a raw image tensor

The exported function accepts `uint8` images of shape `[1, H, W, 3]` and returns:
- `detection_boxes` — `[1, N, 4]` in [ymin, xmin, ymax, xmax] normalized format
- `detection_scores` — `[1, N]` confidence scores in [0, 1]
- `detection_classes` — `[1, N]` integer class IDs
- `num_detections` — `[1]` number of valid detections

In [ ]:
import os

RESEARCH_DIR = "/tmp/models/research"
TRAINING_DIR = "/tmp/training"
EXPORT_DIR   = "/tmp/exported_model"

os.makedirs(EXPORT_DIR, exist_ok=True)

# exporter_main_v2.py reads the *latest* checkpoint from model_dir automatically.
# input_type="image_tensor" means the serving signature accepts a uint8 image tensor.
!python {RESEARCH_DIR}/object_detection/exporter_main_v2.py \
    --input_type=image_tensor \
    --pipeline_config_path={PIPELINE_CONFIG} \
    --trained_checkpoint_dir={TRAINING_DIR} \
    --output_directory={EXPORT_DIR}

SAVED_MODEL_DIR = os.path.join(EXPORT_DIR, "saved_model")
print("\n✅ SavedModel exported to:", SAVED_MODEL_DIR)

# Show what was created
for root, dirs, files in os.walk(EXPORT_DIR):
    level = root.replace(EXPORT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = ' ' * 2 * (level + 1)
    for file in files:
        fpath = os.path.join(root, file)
        print(f'{sub_indent}{file}  ({os.path.getsize(fpath)/1e6:.1f} MB)')

---
## 1️⃣1️⃣ Test Inference on Images

We load the exported SavedModel and run inference on images from the test split.
This confirms the model produces sensible bounding boxes before we convert to TFLite.

### How inference works

1. Load the SavedModel's `"serving_default"` signature
2. Decode a JPEG → uint8 tensor of shape `[H, W, 3]`
3. Add a batch dimension → `[1, H, W, 3]`
4. Call the detection function
5. Filter detections by score threshold (e.g., 0.5)
6. Convert normalized box coordinates back to pixel coordinates
7. Draw boxes using `visualization_utils`

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from object_detection.utils import label_map_util
from object_detection.utils import visualization_utils as viz_utils

# ── Load the exported SavedModel ───────────────────────────────────────────────
SAVED_MODEL_DIR = "/tmp/exported_model/saved_model"
print("Loading SavedModel...")
detect_fn = tf.saved_model.load(SAVED_MODEL_DIR)
print("✅ Model loaded.")

# ── Load category index ────────────────────────────────────────────────────────
# category_index is a dict: {1: {'id': 1, 'name': 'human'}, ...}
# Used by visualization_utils to display class names on the bounding boxes.
category_index = label_map_util.create_category_index_from_labelmap(
    LABEL_MAP_PATH, use_display_name=True
)

In [ ]:
# ── Extract a few test images from the test TFRecord ──────────────────────────
# We decode the raw TFRecord examples to get numpy image arrays for inference.

FEATURE_DESCRIPTION = {
    "image/encoded"    : tf.io.FixedLenFeature([], tf.string),
    "image/filename"   : tf.io.FixedLenFeature([], tf.string, default_value=b""),
}

def decode_image(record_bytes):
    parsed = tf.io.parse_single_example(record_bytes, FEATURE_DESCRIPTION)
    image = tf.image.decode_jpeg(parsed["image/encoded"], channels=3)
    return image.numpy(), parsed["filename"].numpy().decode("utf-8")

# Take the first 3 test images
test_dataset = tf.data.TFRecordDataset(TEST_RECORD).take(3)
test_images  = [decode_image(raw) for raw in test_dataset]

print(f"Loaded {len(test_images)} test image(s) for inference.")

In [ ]:
# ── Run detection and visualize ───────────────────────────────────────────────
SCORE_THRESHOLD = 0.50  # Only show detections with confidence ≥ 50%

fig, axes = plt.subplots(1, len(test_images), figsize=(6 * len(test_images), 6))
if len(test_images) == 1:
    axes = [axes]

for ax, (image_np, filename) in zip(axes, test_images):
    # Add batch dimension: [H, W, 3] → [1, H, W, 3]
    input_tensor = tf.convert_to_tensor(image_np)[tf.newaxis, ...]

    # ── Run inference ──────────────────────────────────────────────────────────
    # detect_fn is the loaded SavedModel's __call__ method.
    # It returns a dict with keys: detection_boxes, detection_scores, etc.
    detections = detect_fn(input_tensor)

    # Convert eager tensors to numpy (remove batch dimension with [0])
    boxes    = detections["detection_boxes"][0].numpy()
    scores   = detections["detection_scores"][0].numpy()
    classes  = detections["detection_classes"][0].numpy().astype(int)
    num_dets = int(detections["num_detections"][0])

    # Make a copy so visualization_utils can draw on it
    image_with_boxes = image_np.copy()
    viz_utils.visualize_boxes_and_labels_on_image_array(
        image_with_boxes,
        boxes,
        classes,
        scores,
        category_index,
        use_normalized_coordinates=True,
        max_boxes_to_draw=20,
        min_score_thresh=SCORE_THRESHOLD,
        agnostic_mode=False,
    )

    ax.imshow(image_with_boxes)
    ax.set_title(os.path.basename(filename) or "test image")
    ax.axis("off")

plt.tight_layout()
plt.savefig("/tmp/inference_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Inference results saved to /tmp/inference_results.png")

---
## 1️⃣2️⃣ Convert to TensorFlow Lite

### Why TFLite?

The SavedModel we exported is optimized for server-side inference:
it contains a full TensorFlow runtime and can use all TF ops.

On a **Raspberry Pi** we need something leaner:
- TFLite runtime is ~1 MB vs. ~50 MB for full TF
- TFLite uses a **flat buffer** format (`.tflite`) that can be memory-mapped directly
  — no heap allocations for model loading
- The interpreter supports **hardware delegates**: the Raspberry Pi Coral Edge TPU USB
  accelerator, XNNPACK (NEON SIMD), and the VideoCore GPU (via the MMAL delegate)

### Quantization Options

| Mode | Weights | Activations | Size | Accuracy | Speed on Pi |
|---|---|---|---|---|---|
| **Float32** (baseline) | float32 | float32 | ~22 MB | 100% | 1× |
| **Float16** | float16 | float32 | ~11 MB | ~99% | 1.3× (GPU delegate) |
| **Dynamic-range int8** | int8 | float32 | ~6 MB | ~97% | 2–3× (CPU) |
| **Full int8** | int8 | int8 | ~6 MB | ~96% | 3–4× (Edge TPU) |

**Recommendation for Raspberry Pi (CPU only):** Dynamic-range int8 — best speed/accuracy trade-off without a calibration dataset.

**Recommendation for Raspberry Pi + Coral USB:** Full int8 — required for Edge TPU delegation.

In [ ]:
import os
import tensorflow as tf

SAVED_MODEL_DIR = "/tmp/exported_model/saved_model"
TFLITE_DIR      = "/tmp/tflite_models"
os.makedirs(TFLITE_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
# Option A: Float32 (no quantization) — maximum accuracy, largest model
# ──────────────────────────────────────────────────────────────────────────────
print("Converting Float32 model...")
converter_fp32 = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)

# Allow TensorFlow Select ops — TFOD post-processing uses ops not in the
# TFLite builtin op set (e.g., tf.image.combined_non_max_suppression).
converter_fp32.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,  # fallback for non-standard ops
]

tflite_fp32 = converter_fp32.convert()
fp32_path = os.path.join(TFLITE_DIR, "model_fp32.tflite")
with open(fp32_path, "wb") as f:
    f.write(tflite_fp32)

print(f"  ✅ Float32 TFLite: {len(tflite_fp32)/1e6:.1f} MB → {fp32_path}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Option B: Float16 quantization — weights compressed to 16-bit floats
# ──────────────────────────────────────────────────────────────────────────────
# Float16 quantization halves model size with almost no accuracy loss.
# On ARM CPUs without FP16 hardware (Raspberry Pi ≤ 4), the runtime
# dequantizes to float32 at inference time, so speed may not improve.
# The main benefit is a smaller .tflite file for storage/transfer.
print("Converting Float16 quantized model...")
converter_fp16 = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]  # enable quantization
converter_fp16.target_spec.supported_types = [tf.float16]  # compress weights to fp16
converter_fp16.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]

tflite_fp16 = converter_fp16.convert()
fp16_path = os.path.join(TFLITE_DIR, "model_fp16.tflite")
with open(fp16_path, "wb") as f:
    f.write(tflite_fp16)

print(f"  ✅ Float16 TFLite: {len(tflite_fp16)/1e6:.1f} MB → {fp16_path}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Option C: Dynamic-range Int8 quantization — best CPU speed on Raspberry Pi
# ──────────────────────────────────────────────────────────────────────────────
# Dynamic-range int8 converts weights to 8-bit integers at conversion time.
# Activations are still computed in float32, but weights are dequantized per-layer
# on the fly. This gives 2–3× speedup on ARM CPUs with no calibration data needed.
print("Converting Dynamic-range Int8 quantized model...")
converter_int8 = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
# Do NOT set supported_types = [tf.int8] here — that triggers full int8 mode
# which requires a representative dataset. We want dynamic-range only.
converter_int8.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]

tflite_int8 = converter_int8.convert()
int8_path = os.path.join(TFLITE_DIR, "model_dynamic_int8.tflite")
with open(int8_path, "wb") as f:
    f.write(tflite_int8)

print(f"  ✅ Dynamic Int8 TFLite: {len(tflite_int8)/1e6:.1f} MB → {int8_path}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Option D: Full Int8 quantization (requires representative dataset)
# ──────────────────────────────────────────────────────────────────────────────
# Full int8 quantizes BOTH weights AND activations to 8-bit integers.
# This is required for Coral Edge TPU delegation which only supports int8.
# It requires a small calibration dataset (~100–500 images) to compute
# activation ranges.
#
# Accuracy trade-off: ~1–3 mAP points lower than float32.
# Speed gain: 3–4× on CPU; up to 40× on Coral Edge TPU.

def representative_dataset():
    """Generator that yields ~100 sample images for int8 calibration.
    
    The converter uses these samples to compute activation min/max statistics
    that determine the int8 quantization scale and zero-point per layer.
    """
    dataset = tf.data.TFRecordDataset(TRAIN_RECORD).take(100)
    FEAT = {"image/encoded": tf.io.FixedLenFeature([], tf.string)}
    for raw in dataset:
        parsed = tf.io.parse_single_example(raw, FEAT)
        image = tf.image.decode_jpeg(parsed["image/encoded"], channels=3)
        # Resize to model input resolution and add batch dim
        image = tf.image.resize(image, [320, 320])
        image = tf.cast(image, tf.uint8)
        yield [image[tf.newaxis, ...]]

print("Converting Full Int8 quantized model (this requires calibration)...")
converter_full_int8 = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter_full_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_full_int8.representative_dataset = representative_dataset
# Force all ops (including activations) to int8
converter_full_int8.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter_full_int8.inference_input_type  = tf.uint8   # input: uint8 image
converter_full_int8.inference_output_type = tf.float32  # keep output as float for post-processing

tflite_full_int8 = converter_full_int8.convert()
full_int8_path = os.path.join(TFLITE_DIR, "model_full_int8.tflite")
with open(full_int8_path, "wb") as f:
    f.write(tflite_full_int8)

print(f"  ✅ Full Int8 TFLite: {len(tflite_full_int8)/1e6:.1f} MB → {full_int8_path}")

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
print("\n📊 TFLite Model Size Summary")
print("-" * 60)
for name, path in [
    ("Float32",        fp32_path),
    ("Float16",        fp16_path),
    ("Dynamic Int8",   int8_path),
    ("Full Int8",      full_int8_path),
]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {name:<20s}: {size_mb:6.1f} MB")
print("-" * 60)

---
## 1️⃣3️⃣ Raspberry Pi Deployment Tips

### 1. Copy the `.tflite` model to your Pi

```bash
# From your laptop / Kaggle
scp /tmp/tflite_models/model_dynamic_int8.tflite pi@raspberrypi.local:~/models/
scp /kaggle/input/datasets/.../train/humans_label_map.pbtxt pi@raspberrypi.local:~/models/
```

### 2. Install TFLite runtime on the Pi

Do NOT install the full `tensorflow` package on the Pi — it is too large.
Install only the runtime:

```bash
# Raspberry Pi OS (64-bit, Python 3.11)
pip install tflite-runtime
```

Or build from the official wheel:
```bash
pip install https://github.com/google-coral/pycoral/releases/download/v2.0.0/tflite_runtime-2.5.0.post1-cp39-cp39-linux_aarch64.whl
```

### 3. Run inference with the TFLite runtime

```python
import cv2
import numpy as np
from tflite_runtime.interpreter import Interpreter

MODEL_PATH    = "models/model_dynamic_int8.tflite"
LABEL_PATH    = "models/humans_label_map.pbtxt"
SCORE_THRESH  = 0.5

# Load the TFLite model and allocate tensors
interpreter = Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

cap = cv2.VideoCapture(0)   # or a video file path

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Preprocess: resize to 320×320, add batch dim
    img = cv2.resize(frame, (320, 320))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = np.expand_dims(img, axis=0)  # shape: (1, 320, 320, 3)

    # Set input tensor and invoke interpreter
    interpreter.set_tensor(input_details[0]['index'], img)
    interpreter.invoke()

    # Read outputs
    boxes   = interpreter.get_tensor(output_details[0]['index'])[0]  # [N, 4]
    classes = interpreter.get_tensor(output_details[1]['index'])[0]  # [N]
    scores  = interpreter.get_tensor(output_details[2]['index'])[0]  # [N]

    h, w, _ = frame.shape
    for i, score in enumerate(scores):
        if score < SCORE_THRESH:
            continue
        ymin, xmin, ymax, xmax = boxes[i]
        # Convert normalized coords to pixel coords
        cv2.rectangle(frame,
                      (int(xmin * w), int(ymin * h)),
                      (int(xmax * w), int(ymax * h)),
                      (0, 255, 0), 2)

    cv2.imshow('Detection', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
```

### 4. Enable XNNPACK delegate for faster CPU inference

```python
from tflite_runtime.interpreter import Interpreter, load_delegate

interpreter = Interpreter(
    model_path=MODEL_PATH,
    experimental_delegates=[load_delegate('libXNNPACK.so')]
)
```

XNNPACK uses ARM NEON SIMD instructions and typically gives **1.5–2× speedup**
over the plain TFLite CPU backend on Raspberry Pi 4.

### 5. Coral Edge TPU (optional)

If you have a Coral USB accelerator plugged in:

```bash
# Install Edge TPU runtime and pycoral library
echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list
curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
sudo apt-get update
sudo apt-get install libedgetpu1-std
pip install pycoral
```

Then use the **full int8** model (the Edge TPU only runs int8) and the EdgeTPU delegate:

```python
from tflite_runtime.interpreter import Interpreter, load_delegate

interpreter = Interpreter(
    model_path='model_full_int8.tflite',
    experimental_delegates=[load_delegate('libedgetpu.so.1')]
)
```

Expected throughput: **~200 FPS** for SSD MobileNet V2 320×320 on Coral USB.

### 6. Performance benchmarks (approximate)

| Device | Model | FPS |
|---|---|---|
| Raspberry Pi 4 (CPU, float32) | ssd_mv2_fpnlite_320 | ~3–5 FPS |
| Raspberry Pi 4 (XNNPACK, int8) | ssd_mv2_fpnlite_320 | ~8–12 FPS |
| Raspberry Pi 4 + Coral USB (int8) | ssd_mv2_fpnlite_320 | ~30–50 FPS |
| Raspberry Pi 5 (CPU, int8) | ssd_mv2_fpnlite_320 | ~15–20 FPS |

---

## 🎯 Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `TypeError: Descriptors cannot not be created directly` | protobuf ≥ 4 | `pip install protobuf==3.20.3` |
| `ModuleNotFoundError: No module named 'object_detection'` | TFOD not installed | Re-run section 5 |
| CUDA OOM during training | Batch size too large | Reduce `batch_size` to 4 |
| mAP stays near 0 after many steps | Wrong `num_classes` | Check label map and `pipeline.config` |
| `ValueError: Could not find checkpoint` on resume | Wrong `TRAINING_DIR` | Verify `TRAINING_DIR` contains `ckpt-*.index` files |
| TFLite conversion fails on NMS op | Missing SELECT_TF_OPS | Add `tf.lite.OpsSet.SELECT_TF_OPS` to supported_ops |
| No detections at inference | Threshold too high | Lower `SCORE_THRESHOLD` to 0.3 |

---

*Happy training! 🤖🍓*